In [14]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# from features import dataviz_structure_categorical, dataviz_structure_numerical, conversion_rate_chart
from utilities import get_path, read_data_from_database

In [15]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# def graficar_tc_bivariada(col_list, dataframe, orden=None):
#   tc = dataframe.groupby(col_list)['y'].mean().to_frame().reset_index()

#   # Gráfica
#   plt.figure(figsize=(13,7))
#   ax = sns.pointplot(x=tc['y'], y=tc[col_list[0]], hue=tc[col_list[1]], order=orden)
#   ax.yaxis.grid(True)
#   ax.xaxis.grid(True)
#   plt.title(f'Tasa de conversión para {col_list[0]} y {col_list[1]}')
#   plt.xlabel('Tasa de conversión (%)')
#   plt.xlim((0,1))

# Environment settings

In [16]:
pd.options.display.max_columns = None  # Remove "dots" from display when printing dataframes

In [17]:
PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/preprocessing/BANCO_BOGOTA.db'

# Read data

In [18]:
df = read_data_from_database(DATABASE_FOLDER, 'clear_train_data')

df.head()

,ID,Edad,Tipo_Trabajo,Estado_Civil,Educacion,mora,Vivienda,Consumo,Contacto,Mes,Dia,Campana,Dias_Ultima_Camp,No_Contactos,Resultado_Anterior,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,1,57,servicios,casado,bachillerato,NaN,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
1,2,37,servicios,casado,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
2,3,40,administrador negocio,casado,primaria,0.0,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
3,4,56,servicios,casado,bachillerato,0.0,0.0,1.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
4,7,25,servicios,soltero,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0


# vizualization

## Categorical data

In [19]:
cols_cat = [
    'Tipo_Trabajo', 'Estado_Civil', 'Educacion', 'mora',
    'Vivienda', 'Consumo', 'Contacto', 'Mes', 'Dia', 'Campana',
    'Resultado_Anterior', 'nr_employed'
]
fig = make_subplots(rows=len(cols_cat), cols=1, subplot_titles=cols_cat)
row, col = 1, 1
for column in cols_cat:
    df_new = df[column].value_counts()
    df_new = df_new.rename_axis(column)
    df_new = df_new.reset_index(name='Count')

    fig.append_trace(go.Bar(
        x=df_new[column],
        y=df_new['Count']
    ), row=row, col=col)
    row += 1
fig.update_layout(height=2600, width=750, title_text='Variables categóricas')
fig.show()

## Numerical data

In [20]:
numerical_columns_histogram = [
    'No_Contactos'
]

fig = go.Figure()
for column in numerical_columns_histogram:
    fig.add_trace(go.Histogram(x=df[column], name=column))
fig.update_layout(barmode='stack')
fig.show()

# Univariate analysis

In [21]:
df.columns

Index(['ID', 'Edad', 'Tipo_Trabajo', 'Estado_Civil', 'Educacion', 'mora',
       'Vivienda', 'Consumo', 'Contacto', 'Mes', 'Dia', 'Campana',
       'Dias_Ultima_Camp', 'No_Contactos', 'Resultado_Anterior',
       'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m',
       'nr_employed', 'y'],
      dtype='object')

In [24]:
predictor_var = 'Edad'
df_group = df.groupby(predictor_var)['y'].mean().mul(100).rename('conversion_rate').reset_index()


fig = px.line(df_group, x='Edad', y='conversion_rate',
                title=f'Tasa de conversión para la variable "{predictor_var}"'
                )
fig.show()

df.loc[:,'groups_ages'] = '18-20'
df.loc[df['Edad']>60,'groups_ages'] = '>60'
df.loc[(df['Edad']>50)&(df['Edad']<=60), 'groups_ages'] = '51-60'
df.loc[(df['Edad']>40)&(df['Edad']<=50), 'groups_ages'] = '41-50'
df.loc[(df['Edad']>30)&(df['Edad']<=40), 'groups_ages'] = '31-40'
df.loc[(df['Edad']>20)&(df['Edad']<=30), 'groups_ages'] = '20-30'

# Y grafiquemos la tasa de conversión para esta nueva columna
df_group = df.groupby('groups_ages')['y'].mean().mul(100).rename('conversion_rate').reset_index()

fig = px.bar(df_group, x='groups_ages', y='conversion_rate',
                     title=f'Tasa de conversión para la variable "{predictor_var}"'
                     )
fig.show()

# Vibariate analysis

In [ ]:
cr_bivariate(['Contact_hour_round','Contact_channel'], df)